# Quick boosted-tree model comparison

This notebook compares XGBoost, LightGBM, and CatBoost on the full chronological training and validation partitions. The linear-model workflow is in `linear_modeling.ipynb`. Install the optional packages before running:

```powershell
python -m pip install xgboost lightgbm catboost
```

The test partition is not used in this quick comparison.

In [9]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from boosted_preprocessing import make_boosted_tree_preprocessor, prepare_boosted_tree_features
from model_data import RAW_MODEL_COLUMNS
from model_evaluation import macro_f1_test
from prep_job import add_split_time, add_target, chronological_split, load_monthly_data, preprocess_partitions

# Use all rows from the chronological training and validation partitions.

In [10]:
raw_data = load_monthly_data(PROJECT_ROOT / "data", RAW_MODEL_COLUMNS)
labelled_data = add_target(raw_data)
split_data = chronological_split(add_split_time(labelled_data))
processed_data = preprocess_partitions(split_data)

train_features, y_train = prepare_boosted_tree_features(processed_data["train"])
validation_features, y_validation = prepare_boosted_tree_features(processed_data["validation"])

preprocessor = make_boosted_tree_preprocessor()
X_train = preprocessor.fit_transform(train_features)
X_validation = preprocessor.transform(validation_features)

target_names = sorted(y_train.unique())
target_to_id = {name: index for index, name in enumerate(target_names)}
y_train_encoded = y_train.map(target_to_id)
y_validation_encoded = y_validation.map(target_to_id)

pd.DataFrame({
    "partition": ["train", "validation"],
    "rows": [len(y_train_encoded), len(y_validation_encoded)],
    "features": [X_train.shape[1], X_validation.shape[1]],
})

,partition,rows,features
0,train,9864275,8
1,validation,2120356,8


## Train and compare the boosted models

All three models use the same preprocessing and chronological validation sample. Macro-F1 is the comparison metric. GPU execution is attempted first; each model falls back to CPU if its GPU backend is unavailable.

In [11]:
from catboost import CatBoostClassifier, CatBoostError
from lightgbm import LGBMClassifier
from lightgbm.basic import LightGBMError
from xgboost import XGBClassifier
from xgboost.core import XGBoostError

categorical_columns = [
    column for column in X_train.columns
    if str(X_train[column].dtype) == "category"
]
categorical_indices = [X_train.columns.get_loc(column) for column in categorical_columns]
USE_GPU = True

models = {
    "xgboost": XGBClassifier(
        objective="multi:softmax",
        num_class=len(target_names),
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        tree_method="hist",
        device="cuda",
        enable_categorical=True,
        n_jobs=-1,
        random_state=42,
    ),
    "lightgbm": LGBMClassifier(
        objective="multiclass",
        num_class=len(target_names),
        n_estimators=100,
        num_leaves=31,
        learning_rate=0.1,
        class_weight="balanced",
        verbosity=-1,
        device_type="cpu",
        random_state=42,
    ),
    "catboost": CatBoostClassifier(
        loss_function="MultiClass",
        iterations=100,
        depth=6,
        learning_rate=0.1,
        auto_class_weights="Balanced",
        task_type="GPU",
        verbose=False,
        random_seed=42,
    ),
}

results = []
for name, model in models.items():
    try:
        if name == "lightgbm":
            model.fit(X_train, y_train_encoded, categorical_feature=categorical_columns)
        elif name == "catboost":
            model.fit(X_train, y_train_encoded, cat_features=categorical_indices)
        else:
            model.fit(X_train, y_train_encoded)
    except (XGBoostError, LightGBMError, CatBoostError) as error:
        if not USE_GPU:
            raise
        print(f"{name} GPU unavailable; retrying on CPU: {error}")
        if name == "xgboost":
            model.set_params(device="cpu")
        elif name == "lightgbm":
            model.set_params(device_type="cpu")
        else:
            model.set_params(task_type="CPU")
        if name == "lightgbm":
            model.fit(X_train, y_train_encoded, categorical_feature=categorical_columns)
        elif name == "catboost":
            model.fit(X_train, y_train_encoded, cat_features=categorical_indices)
        else:
            model.fit(X_train, y_train_encoded)

    if name == "xgboost":
        # Validation data is a pandas/CPU frame; avoid XGBoost device mismatch.
        model.set_params(device="cpu")

    results.append(
        macro_f1_test(
            model,
            X_validation,
            y_validation_encoded,
            model_name=name,
        )
    )

pd.DataFrame(results).sort_values(
    ["macro_f1", "accuracy"], ascending=False
).reset_index(drop=True)

,model,n_test,macro_f1,accuracy
0,lightgbm,2120356,0.325058,0.394124
1,catboost,2120356,0.311368,0.385897
2,xgboost,2120356,0.259233,0.476203


### Modelling conclusion

On the validation set, LightGBM achieved the highest macro-F1 (0.3251), followed by CatBoost (0.3114) and logistic regression (0.3093). LightGBM therefore improves on the logistic-regression reference by 0.0158 macro-F1, while CatBoost is only slightly higher by 0.0021. The boosted models provide a stronger nonlinear alternative, with LightGBM currently the preferred model for final test evaluation.